In [ ]:
# This script rounds and organizes final static attributes for catchments.
# It reads data from an excel file containing the final values
# The processed data is then saved as separate CSV files for each category.

In [1]:
import pandas as pd
import os

In [2]:
path = 'CAMELS-FI construction/CAMELS-FI_static_attributes.ods'
full_data = pd.read_excel(path)

In [3]:
full_data

,gauge_id,area,crop_perc_2000,grass_perc_2000,shrub_perc_2000,dwood_perc_2000,ewood_perc_2000,urban_perc_2000,inwater_perc_2000,bares_perc_2000,...,frac_snow,high_prec_freq,high_prec_dur,high_prec_timing,low_prec_freq,low_prec_dur,low_prec_timing,ice_correction,owner_id,owner_name
0,896,379.328090,14.057020,0.284605,16.789687,5.296422,46.125775,4.629865,9.378068,0.171887,...,0.265468,15.699284,1.162963,jja,229.022883,3.921804,mam,yes,16,SYKE
1,905,858.070423,2.309383,0.152469,21.175889,1.726141,62.810607,0.815175,5.207016,0.184762,...,0.302964,14.432675,1.148541,jja,221.289902,3.561695,mam,yes,16,SYKE
2,907,412.224485,4.273159,0.158796,26.568805,2.649461,58.125275,1.085481,1.675725,0.139795,...,0.293810,15.665952,1.135266,jja,228.122924,3.808570,mam,yes,16,SYKE
3,908,1043.365363,6.202935,0.360695,24.261652,1.907429,57.465996,1.734754,4.072438,0.204679,...,0.291571,15.432629,1.118357,jja,229.156210,3.860191,mam,no,2,Other
4,923,215.413012,1.913932,0.067483,20.544114,1.646585,62.836410,1.422024,3.853708,0.371013,...,0.300552,15.165975,1.104369,jja,220.356612,3.709877,mam,yes,16,SYKE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,3727,25.802921,0.045317,0.000000,22.741909,0.140721,25.403678,0.667827,21.439645,0.040547,...,0.341916,16.199261,1.154394,jja,235.289264,3.897846,mam,no,2,Other
316,3728,64.292793,0.097630,0.140702,27.133504,0.848042,29.207665,0.838470,17.449941,0.049772,...,0.343298,16.565911,1.150463,jja,236.589204,3.936772,mam,no,2,Other
317,3850,150.749117,20.143618,0.249438,16.550603,1.633266,44.195668,11.783185,0.944119,2.171304,...,0.241555,16.632574,1.202410,jja,237.355836,4.206143,NaN,yes,7,ELY Southwest Finland
318,3907,143.112642,22.684787,0.133094,15.555604,0.833567,51.185532,5.124992,0.043212,0.694423,...,0.174382,19.965756,1.160853,jja,247.322048,4.223108,mam,no,4,ELY Varsinais-Suomi


# Rounding the values to a reasonable amount of decimal places

In [4]:
# coordinates should not be rounded
lon_lat = full_data[['gauge_lat', 'gauge_lon']]

full_data = full_data.round(2)
full_data[['gauge_lat', 'gauge_lon']] = lon_lat

In [5]:

table_columns = {}

dst_dir = '/media/iielse/T9/CAMELS-FI/data'

columns = ['gauge_id'] + [
    'gauge_name', 'owner_id', 'owner_name',
    'gauge_lon','gauge_lat', 'gauge_easting',
    'gauge_northing', 'area', 'nestedness',
    'basin_id', 'basin_name','water_region_code',
    'water_region_name', 'cross_border_perc','reference_gauge',
    'ice_correction'
    ]
table_columns['meta'] = columns

columns = ['gauge_id'] + [
    'slope', 'elev_gauge', 'elev_mean', 'elev_min',
    'elev_10', 'elev_50', 'elev_90',
    'elev_max', 'elev_range'
    ]
table_columns['topographic'] = columns

columns = ['gauge_id'] + [
    'p_mean', 'pet_mean', 'temperature_mean',
    'aridity', 'p_seasonality', 'frac_snow',
    'high_prec_freq', 'high_prec_dur', 'high_prec_timing', 
    'low_prec_freq', 'low_prec_dur', 'low_prec_timing'
    ]
table_columns['climatic'] = columns

columns = ['gauge_id'] + [
    'p_mean', 'pet_mean', 'temperature_mean',
    'aridity', 'p_seasonality', 'frac_snow',
    'high_prec_freq', 'high_prec_dur', 'high_prec_timing', 
    'low_prec_freq', 'low_prec_dur', 'low_prec_timing'
    ]
table_columns['climatic'] = columns

columns = ['gauge_id'] + [
    'timeseries_number_of_years', 'sign_start_date', 'sign_end_date', 
    'sign_number_of_years', 'sign_number_of_obs', 'q_mean', 
    'runoff_ratio', 'stream_elas', 'slope_fdc',
    'baseflow_index_ladson', 'baseflow_index_lfstat',  'hfd_mean',
    'Q5', 'Q95', 'high_q_freq',
    'high_q_dur', 'low_q_freq', 'low_q_dur',
    'zero_q_freq'
    ]
table_columns['hydrologic'] = columns

columns = ['gauge_id'] + [
    'bedrock_perc', 'coarse_perc', 'silt_perc',
    'till_perc', 'clay_perc', 'peat_perc', 
    'soil_depth'
    ]
table_columns['soil'] = columns

#all available Corines are used
lc_classes = [
    'crop_perc', 'grass_perc', 'shrub_perc',
    'dwood_perc', 'ewood_perc', 'urban_perc',
    'inwater_perc', 'bares_perc', 'wetland_perc'
    ]
lc_years = [2000, 2006, 2012, 2018]

lc_columns = [f"{lc_class}_{year}" for year in lc_years for lc_class in lc_classes]

columns = ['gauge_id'] + lc_columns
table_columns['landcover'] = columns

columns = ['gauge_id'] + [
    'num_inhabitants', 'dens_inhabitants', 'num_dam',
    'num_reservoir', 'reservoir_cap', 'num_regulation_other',
    'regulation_level'
    ]
table_columns['humaninfluence'] = columns


for name in table_columns:
    data = full_data[table_columns[name]]
    dst_path = os.path.join(dst_dir, f"CAMELS_FI_{name}_attributes.csv")
    data.to_csv(dst_path, sep=',', index=False)